# AI 302 - Train Your Own Small LLM: Solution Notebook

Run top-to-bottom in Google Colab (Runtime -> Change runtime type -> **T4 GPU**).

This notebook contains every code cell of the lab, in order. It builds, pre-trains and runs a small
GPT-style language model from scratch on the Tiny Shakespeare corpus, then demonstrates sampling.
See the lab content for the explanations behind each step:
https://fortinetcloudcse.github.io/genai-creator-lab2/


## Phase 0 - Introduction & Environment


### Prepare the Environment


In [ ]:
#@title Setup dependencies
import os
import time
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from keras import layers, ops

print("TensorFlow:", tf.__version__)
print("Keras     :", keras.__version__)

In [ ]:
#@title Check GPU
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("✅ GPU available:", gpus[0].name)
else:
    print("⚠️ No GPU detected. Training will still work, but roughly 5-10x slower.")

## Phase 1 - Tokenization & Embeddings


### Load the Text Corpus


In [ ]:
#@title Load the Tiny Shakespeare corpus
CORPUS_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

corpus_path = keras.utils.get_file("tinyshakespeare.txt", CORPUS_URL)
with open(corpus_path, "r", encoding="utf-8") as f:
    text = f.read()

print("Corpus length (characters):", len(text))
print("\n--- First 300 characters ---\n")
print(text[:300])

In [ ]:
#@title Inspect the character vocabulary
chars = sorted(set(text))
vocab_size = len(chars)

print("Number of unique characters (vocab size):", vocab_size)
print("Characters:", "".join(chars))

### Tokenization


In [ ]:
#@title Build the char <-> int mappings
stoi = {ch: i for i, ch in enumerate(chars)}   # string-to-integer
itos = {i: ch for i, ch in enumerate(chars)}   # integer-to-string

def encode(s):
    """Turn a string into a list of token IDs."""
    return [stoi[c] for c in s]

def decode(ids):
    """Turn a list of token IDs back into a string."""
    return "".join(itos[int(i)] for i in ids)

# Quick sanity check
sample = "To be, or not to be"
print("Original:", sample)
print("Encoded :", encode(sample))
print("Decoded :", decode(encode(sample)))

In [ ]:
#@title Encode the full corpus
data = np.array(encode(text), dtype=np.int32)

print("Encoded corpus shape:", data.shape)
print("First 50 token IDs :", data[:50])
print("Decoded again      :", repr(decode(data[:50])))

In [ ]:
#@title Train / validation split
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

print("Train tokens:", len(train_data))
print("Val   tokens:", len(val_data))

### Embeddings & Positional Encoding


In [ ]:
#@title See what an embedding layer does
demo_embed_dim = 64          # size of each token's vector

demo_embedding = layers.Embedding(input_dim=vocab_size, output_dim=demo_embed_dim)

example_ids = np.array([encode("To be")])     # 5 characters: T, o, space, b, e
example_vectors = demo_embedding(example_ids)

print("Input IDs shape :", example_ids.shape)        # (1, 5)
print("Embedded shape  :", example_vectors.shape)    # (1, 5, 64)

In [ ]:
#@title Combined token + position embedding layer
block_size = 128   # maximum context length the model can see at once

@keras.saving.register_keras_serializable()
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, vocab_size, block_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.embed_dim  = embed_dim
        self.token_emb = layers.Embedding(vocab_size, embed_dim)
        self.pos_emb   = layers.Embedding(block_size, embed_dim)

    def build(self, input_shape):
        self.token_emb.build(input_shape)
        self.pos_emb.build(input_shape)
        super().build(input_shape)

    def call(self, ids):
        seq_len = ops.shape(ids)[1]
        positions = ops.arange(0, seq_len, 1)
        return self.token_emb(ids) + self.pos_emb(positions)

    def get_config(self):
        config = super().get_config()
        config.update(vocab_size=self.vocab_size,
                      block_size=self.block_size,
                      embed_dim=self.embed_dim)
        return config


# Quick check
embedding_demo = TokenAndPositionEmbedding(vocab_size, block_size, demo_embed_dim)
print("Position-aware embedding shape:", embedding_demo(example_ids).shape)   # (1, 5, 64)

## Phase 2 - Attention & the Transformer


### Self-Attention Intuition


In [ ]:
#@title Scaled dot-product attention, by hand
def scaled_dot_product_attention(q, k, v, mask=None):
    """q, k, v shapes: (batch, seq_len, d_k). Returns (output, attention_weights)."""
    d_k = tf.cast(tf.shape(k)[-1], tf.float32)
    scores = tf.matmul(q, k, transpose_b=True) / tf.math.sqrt(d_k)   # (batch, seq, seq)
    if mask is not None:
        scores += (mask * -1e9)        # push masked positions to -infinity before softmax
    weights = tf.nn.softmax(scores, axis=-1)
    return tf.matmul(weights, v), weights


# Run it on 4 random tokens and look at the attention weights
demo = tf.random.normal((1, 4, 8))
out, w = scaled_dot_product_attention(demo, demo, demo)
print("Output shape           :", out.shape)          # (1, 4, 8) - same shape as the input
print("Attention weight matrix:\n", np.round(w[0].numpy(), 2))
print("Every row sums to 1    :", np.allclose(w[0].numpy().sum(axis=-1), 1.0))

### Causal Masking - Why It Can't See the Future


In [ ]:
#@title Build a causal mask and see its effect
def causal_mask(seq_len):
    """Returns a (seq_len, seq_len) matrix: 1 where attention must be BLOCKED."""
    return 1.0 - np.tril(np.ones((seq_len, seq_len), dtype="float32"))

print("Blocked positions (1 = blocked):")
print(causal_mask(5).astype(int))

# Apply it to the hand-written attention from the previous page
demo = tf.random.normal((1, 5, 8))
_, masked_weights = scaled_dot_product_attention(demo, demo, demo, mask=causal_mask(5))
print("\nAttention weights with the mask applied:")
print(np.round(masked_weights[0].numpy(), 2))

### Building the Decoder Block


In [ ]:
#@title Define a single decoder (Transformer) block
@keras.saving.register_keras_serializable()
class DecoderBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim    = embed_dim
        self.num_heads    = num_heads
        self.ff_dim       = ff_dim
        self.dropout_rate = dropout_rate

        self.attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,   # split the vector across the heads
        )
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="gelu"),
            layers.Dense(embed_dim),
        ])
        self.ln1   = layers.LayerNormalization(epsilon=1e-6)
        self.ln2   = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout_rate)
        self.drop2 = layers.Dropout(dropout_rate)

    def build(self, input_shape):
        self.ln1.build(input_shape)
        self.ln2.build(input_shape)
        self.attn.build(input_shape, input_shape, input_shape)
        self.ffn.build(input_shape)
        super().build(input_shape)

    def call(self, x, training=False):
        # 1) masked self-attention + residual
        h = self.ln1(x)
        attn_out = self.attn(query=h, value=h, key=h,
                             use_causal_mask=True,      # the triangular mask from the last page
                             training=training)
        x = x + self.drop1(attn_out, training=training)

        # 2) feed-forward + residual
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.drop2(ffn_out, training=training)
        return x

    def get_config(self):
        config = super().get_config()
        config.update(embed_dim=self.embed_dim,
                      num_heads=self.num_heads,
                      ff_dim=self.ff_dim,
                      dropout_rate=self.dropout_rate)
        return config


# Sanity check: the block must return exactly the shape it was given
demo_block = DecoderBlock(embed_dim=64, num_heads=4, ff_dim=256)
print("In :", (2, 10, 64))
print("Out:", demo_block(tf.random.normal((2, 10, 64))).shape)

## Phase 3 - The Training Pipeline


### Next-Token Prediction & Training Data


In [ ]:
#@title Build (input, target) pairs from the corpus
batch_size = 64          # how many windows per training step
# block_size = 128 was already defined with the embedding layer

def get_batch(source, batch_size, block_size):
    """Cut batch_size random windows out of `source`; y is x shifted one token right."""
    starts = np.random.randint(0, len(source) - block_size - 1, size=batch_size)
    x = np.stack([source[i     : i + block_size    ] for i in starts])
    y = np.stack([source[i + 1 : i + block_size + 1] for i in starts])   # shifted by 1
    return tf.constant(x), tf.constant(y)

xb, yb = get_batch(train_data, batch_size, block_size)
print("Input  batch shape:", xb.shape)   # (64, 128)
print("Target batch shape:", yb.shape)   # (64, 128)
print("\nFirst example, decoded input :", repr(decode(xb[0][:40].numpy())))
print("First example, decoded target:", repr(decode(yb[0][:40].numpy())))

### Assemble & Compile the Model


In [ ]:
#@title Hyperparameters - these are your tuning knobs for the challenge
embed_dim     = 64      # token/position vector size  (block_size = 128 from Phase 1)
num_heads     = 4       # parallel attention heads    (must divide embed_dim)
ff_dim        = 256     # feed-forward width inside each block
num_blocks    = 4       # how many decoder blocks to stack
dropout_rate  = 0.1
learning_rate = 1e-3

In [ ]:
#@title Build the GPT-style model
def build_gpt():
    inputs = keras.Input(shape=(None,), dtype="int32")

    x = TokenAndPositionEmbedding(vocab_size, block_size, embed_dim)(inputs)

    for _ in range(num_blocks):
        x = DecoderBlock(embed_dim, num_heads, ff_dim, dropout_rate)(x)

    x = layers.LayerNormalization(epsilon=1e-6)(x)
    logits = layers.Dense(vocab_size)(x)        # one score per vocabulary token
    return keras.Model(inputs=inputs, outputs=logits)

model = build_gpt()
model.summary()

In [ ]:
#@title Compile the model
loss_fn   = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

model.compile(optimizer=optimizer, loss=loss_fn)
print("Model compiled. Ready to train.")

### Train the Model


In [ ]:
#@title Validation loss helper
@tf.function
def val_loss_step(x, y):
    return loss_fn(y, model(x, training=False))

def estimate_val_loss(iters=20):
    losses = [float(val_loss_step(*get_batch(val_data, batch_size, block_size)))
              for _ in range(iters)]
    return float(np.mean(losses))

In [ ]:
#@title Train the model
steps      = 3000        # number of training steps  (a tuning knob)
eval_every = 500

@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        logits = model(x, training=True)
        loss = loss_fn(y, logits)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

history = []
start = time.time()

for step in range(1, steps + 1):
    xb, yb = get_batch(train_data, batch_size, block_size)
    loss = train_step(xb, yb)

    if step == 1 or step % eval_every == 0:
        vl = estimate_val_loss()
        history.append((step, float(loss), vl))
        print(f"step {step:5d} | train {float(loss):.4f} | val {vl:.4f} "
              f"| perplexity {np.exp(vl):5.1f} | {time.time()-start:6.1f}s")

print("\n✅ Training complete.")

In [ ]:
#@title Plot the training and validation loss
steps_x  = [h[0] for h in history]
train_y  = [h[1] for h in history]
val_y    = [h[2] for h in history]

plt.plot(steps_x, train_y, marker="o", label="train")
plt.plot(steps_x, val_y,   marker="o", label="validation")
plt.title("Loss (next-token prediction)")
plt.xlabel("Step")
plt.ylabel("Cross-entropy loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#@title Save the trained model
model.save("genai_foundation.keras")
print("[*] Saved as genai_foundation.keras")

In [ ]:
#@title Reload it and confirm the weights survived
reloaded = keras.models.load_model("genai_foundation.keras")

xb, yb = get_batch(val_data, batch_size, block_size)
print("Original loss:", float(loss_fn(yb, model(xb, training=False))))
print("Reloaded loss:", float(loss_fn(yb, reloaded(xb, training=False))))

## Phase 4 - Inference & Sampling


### Inference - Generating Text Token by Token


In [ ]:
#@title Generate text from the trained model
def generate(model, prompt, max_new_tokens=300, temperature=1.0, top_k=None, top_p=None):
    ids = encode(prompt)

    for _ in range(max_new_tokens):
        # the model can only ever see the last block_size tokens
        context = tf.constant([ids[-block_size:]], dtype=tf.int32)
        logits = np.array(model(context, training=False))[0, -1, :].astype("float64")

        # temperature 0 means "no randomness at all": always take the top token
        if temperature <= 0:
            ids.append(int(np.argmax(logits)))
            continue

        logits = logits / temperature

        if top_k is not None:                       # keep only the k highest-scoring tokens
            kth_best = np.sort(logits)[-top_k]
            logits[logits < kth_best] = -np.inf

        probs = np.exp(logits - logits.max())       # softmax, numerically safe
        probs /= probs.sum()

        if top_p is not None:                       # keep the smallest set summing to p
            order = np.argsort(probs)[::-1]
            cutoff = np.searchsorted(np.cumsum(probs[order]), top_p) + 1
            keep = np.zeros_like(probs)
            keep[order[:cutoff]] = 1.0
            probs *= keep
            probs /= probs.sum()

        ids.append(int(np.random.choice(len(probs), p=probs)))

    return decode(ids)

In [ ]:
#@title First generation
print(generate(model, prompt="ROMEO:", max_new_tokens=400, temperature=0.8))

### Sampling - Temperature, Top-k, Top-p


In [ ]:
#@title Compare sampling settings
settings = [
    ("greedy (temperature=0)",   dict(temperature=0.0)),
    ("focused (0.2)",            dict(temperature=0.2)),
    ("balanced (0.8)",           dict(temperature=0.8)),
    ("wild (1.4)",               dict(temperature=1.4)),
    ("0.8 + top_k=20",           dict(temperature=0.8, top_k=20)),
    ("0.8 + top_p=0.9",          dict(temperature=0.8, top_p=0.9)),
]

for label, kwargs in settings:
    print("=" * 70)
    print(f"### {label}")
    print(generate(model, "ROMEO:", max_new_tokens=200, **kwargs))

## Phase 6 - Final Challenge


### Final Challenge


In [ ]:
#@title Score your model
val = estimate_val_loss(iters=50)
print(f"Validation loss : {val:.4f}")
print(f"Perplexity      : {np.exp(val):.2f}")

In [ ]:
#@title Produce your final report
print("Validation loss :", round(estimate_val_loss(iters=50), 4))
print("Hyperparameters :", dict(block_size=block_size, embed_dim=embed_dim,
                                num_heads=num_heads, ff_dim=ff_dim,
                                num_blocks=num_blocks, dropout_rate=dropout_rate,
                                learning_rate=learning_rate, batch_size=batch_size,
                                steps=steps))
print("Parameters      :", model.count_params())
print("\n--- Sample ---\n")
print(generate(model, "ROMEO:", max_new_tokens=400, temperature=0.8, top_k=40))